## CONFORMAL PREDICTION

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from mineral_id.conformal import (
    calibration_scores,
    finite_sample_quantile,
    prediction_sets
)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

with np.load(
    PROJECT_ROOT / "data" / "processed" /
    "primary_532nm_features.npz"
) as data:

    X_l2_full = data["X_l2"]
    y_full = data["mineral"].astype(str)
    ids_full = data["rruff_id"].astype(str)

split_assignment = pd.read_csv(
    PROJECT_ROOT / "data" / "manifests" /
    "primary_532nm_split_assignment.csv"
)

index_map = {
    sample_id: i
    for i, sample_id in enumerate(ids_full)
}

def get_partition(name):

    partition_ids = split_assignment.loc[
        split_assignment["split"] == name,
        "rruff_id"
    ].to_numpy()

    indices = np.array([
        index_map[sample_id]
        for sample_id in partition_ids
    ])

    return (
        X_l2_full[indices],
        y_full[indices],
        ids_full[indices]
    )

X_train_final, y_train_final, ids_train = get_partition("train")

X_cal, y_cal, ids_cal = get_partition("calibration")

X_test, y_test, ids_test = get_partition("test")

print("Training:", X_train_final.shape)
print("Calibration:", X_cal.shape)
print("Testing:", X_test.shape)

In [ ]:
assert set(ids_train).isdisjoint(ids_cal)
assert set(ids_train).isdisjoint(ids_test)
assert set(ids_cal).isdisjoint(ids_test)

assert len(np.unique(ids_train)) == 82
assert len(np.unique(ids_cal)) == 27
assert len(np.unique(ids_test)) == 28

In [ ]:
partition_summary = pd.DataFrame({

    "train": pd.Series(y_train_final).value_counts(),

    "calibration": pd.Series(y_cal).value_counts(),

    "test": pd.Series(y_test).value_counts()

}).fillna(0).astype(int)

display(partition_summary)

#### 1.1 Fit the final RBF SVM

In [ ]:
# Create group-aware probability-calibration folds
probability_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

probability_splits = list(
    probability_cv.split(
        X_train_final,
        y_train_final,
        groups=ids_train
    )
)

# Base SVM: same selected hyperparameters
base_svm = SVC(
    kernel="rbf",
    C=1000,
    gamma=0.1,
    class_weight="balanced",
    probability=False,
    random_state=42
)

# Calibrate probabilities using training-only cross-validation
final_model = CalibratedClassifierCV(
    estimator=base_svm,
    method="sigmoid",
    cv=probability_splits,
    ensemble=False
)

final_model.fit(
    X_train_final,
    y_train_final
)

print("Final calibrated RBF SVM fitted.")
print("Classes:", final_model.classes_)

In [ ]:
cal_probabilities = final_model.predict_proba(X_cal)

print("Probability matrix:", cal_probabilities.shape)

print(
    "All probabilities finite:",
    np.isfinite(cal_probabilities).all()
)

print(
    "Probability rows sum to one:",
    np.allclose(cal_probabilities.sum(axis=1), 1)
)

In [ ]:
cal_scores = calibration_scores(
    probabilities=cal_probabilities,
    y_true=y_cal,
    classes=final_model.classes_
)

print("Calibration scores:", len(cal_scores))

print(pd.Series(cal_scores).describe())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(
    cal_scores,
    bins=10,
    edgecolor="black"
)

ax.set_title("Calibration nonconformity scores")

ax.set_xlabel("Nonconformity score")

ax.set_ylabel("Number of observations")

ax.grid(alpha=0.2)

plt.tight_layout()

fig.savefig(
    PROJECT_ROOT / "reports" / "figures" /
    "calibration_scores.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
thresholds = {}

for alpha in [0.10, 0.05]:

    q = finite_sample_quantile(
        cal_scores,
        alpha=alpha
    )

    thresholds[alpha] = q

    print(
        f"Target coverage: {1-alpha:.0%}"
    )

    print(
        f"Conformal threshold: {q:.5f}"
    )

#### 1.2 Evaluate on test set

In [ ]:
test_probabilities = final_model.predict_proba(X_test)

test_predictions = final_model.predict(X_test)

print(
    "Test accuracy:",
    accuracy_score(y_test, test_predictions)
)

print(
    "Test Macro-F1:",
    f1_score(
        y_test,
        test_predictions,
        average="macro"
    )
)

print(
    classification_report(
        y_test,
        test_predictions,
        zero_division=0
    )
)

In [ ]:
conformal_results = {}

for alpha, threshold in thresholds.items():

    sets = prediction_sets(
        probabilities=test_probabilities,
        classes=final_model.classes_,
        threshold=threshold
    )

    conformal_results[alpha] = sets

In [ ]:
for alpha, sets in conformal_results.items():

    coverage = np.mean([
        true_label in prediction_set
        for true_label, prediction_set
        in zip(y_test, sets)
    ])

    set_sizes = np.array([
        len(prediction_set)
        for prediction_set in sets
    ])

    singleton_rate = np.mean(set_sizes == 1)

    empty_rate = np.mean(set_sizes == 0)

    print("\n" + "=" * 55)

    print(f"Target coverage: {1-alpha:.0%}")

    print(f"Empirical test coverage: {coverage:.3f}")

    print(f"Average set size: {set_sizes.mean():.3f}")

    print(f"Median set size: {np.median(set_sizes):.1f}")

    print(f"Singleton prediction rate: {singleton_rate:.3f}")

    print(f"Empty prediction rate: {empty_rate:.3f}")

In [ ]:
prediction_table = pd.DataFrame({

    "rruff_id": ids_test,

    "true_mineral": y_test,

    "top_prediction": test_predictions,

    "top_probability": test_probabilities.max(axis=1),

    "set_90": [
        ", ".join(map(str, s))
        for s in conformal_results[0.10]
    ],

    "set_95": [
        ", ".join(map(str, s))
        for s in conformal_results[0.05]
    ]

})

display(
    prediction_table.sort_values(
        "true_mineral"
    )
)

In [ ]:
class_coverage = []

for mineral in np.unique(y_test):

    mask = y_test == mineral

    n_class = mask.sum()

    for alpha, sets in conformal_results.items():

        covered = np.array([
            y_test[i] in sets[i]
            for i in range(len(y_test))
        ])

        class_coverage.append({

            "mineral": mineral,

            "target_coverage": 1 - alpha,

            "n_test": n_class,

            "empirical_coverage": covered[mask].mean(),

        })

class_coverage_df = pd.DataFrame(class_coverage)

display(
    class_coverage_df.round(3)
)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

figures_dir = PROJECT_ROOT / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(11, 9))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    labels=final_model.classes_,
    display_labels=final_model.classes_,
    cmap="Blues",
    values_format="d",
    xticks_rotation=45,
    ax=ax
)

ax.set_title("Final RBF SVM — Held-out test set")

plt.tight_layout()

fig.savefig(
    figures_dir / "test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
summary_records = []

for alpha, sets in conformal_results.items():

    set_sizes = np.array([len(s) for s in sets])

    coverage = np.mean([
        true_label in prediction_set
        for true_label, prediction_set
        in zip(y_test, sets)
    ])

    summary_records.append({
        "target": f"{1-alpha:.0%}",
        "coverage": coverage,
        "mean_set_size": set_sizes.mean(),
        "singleton_rate": np.mean(set_sizes == 1)
    })

summary_df = pd.DataFrame(summary_records)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(
    summary_df["target"],
    summary_df["coverage"]
)

axes[0].set_ylim(0, 1.08)
axes[0].set_title("Empirical test coverage")
axes[0].set_ylabel("Coverage")

axes[1].bar(
    summary_df["target"],
    summary_df["mean_set_size"]
)

axes[1].set_title("Average prediction-set size")
axes[1].set_ylabel("Number of candidate minerals")

for ax in axes:
    ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

fig.savefig(
    figures_dir / "conformal_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()